In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from shapely import wkt
import altair as alt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import shutil
import subprocess
import os
import folium
import folium.plugins
import datetime
alt.data_transformers.disable_max_rows()
import warnings
warnings.filterwarnings("ignore")



os.environ["JAVA_HOME"] = subprocess.run(
    ["/usr/libexec/java_home", "-v", "21"],
    capture_output=True, text=True, check=True
).stdout.strip()


from r5py.util.config import Config
import r5py

In [ ]:
# Importing osmnx

# Define the polygon for the 1st district of Budapest
admin_district = ox.geocode_to_gdf('Àrea Metropolitana de Barcelona, Spain')
admin_district.plot()
admin_poly = admin_district.geometry.values[0]

# Load the graph from OSMnx
G = ox.graph_from_polygon(admin_poly, network_type='walk')

print('Number of intersections: ', G.number_of_nodes())
print('Number of road segments:',  G.number_of_edges())

In [ ]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)


neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).properties(
    width=600,
    height=600
    
).encode(tooltip=['name'])

In [ ]:
bus_sol = pd.read_csv('Nodes/Bus-Sol.csv')
bus = pd.read_csv('Nodes/Bus.csv')
fgc_sol = pd.read_csv('Nodes/FGC-Sol.csv')
fgc = pd.read_csv('Nodes/FGC.csv')
metro_sol = pd.read_csv('Nodes/Metro-Sol.csv')
metro = pd.read_csv('Nodes/Metro.csv')
tram_sol = pd.read_csv('Nodes/Tram-Sol.csv')
tram = pd.read_csv('Nodes/Tram.csv')
destinations = pd.read_csv('Nodes/Destinations.csv')

all_stops = pd.concat([bus_sol, bus, fgc_sol, fgc, metro_sol, metro, tram_sol, tram], ignore_index=True)
all_stops['geometry']= all_stops['geometry'].apply(wkt.loads)
all_stops = gpd.GeoDataFrame(all_stops, geometry='geometry', crs="EPSG:4326")
#all_stops = all_stops[:100]


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

admin_district.plot(
    ax=ax,
    color="lightblue",
    edgecolor="black",
    linewidth=1
)

all_stops.plot(
    ax=ax,
    color="red",
    markersize=8,
    edgecolor="black",
    linewidth=0.3
)

ax.set_title("Àrea Metropolitana de Barcelona i parades")
ax.axis("off")
plt.show()

In [ ]:
walking_speed = 1.39

for u, v, data in G.edges(data=True):
    data["travel_time"] = data["length"] / walking_speed


isochrone_time = 5
isochrone_polys = []

process = 1
for stop in all_stops['stop_id'].unique():
    print(f"Processing stop {stop} ({process}/{len(all_stops['stop_id'].unique())})")
    process += 1
    p = all_stops[all_stops['stop_id'] == stop].iloc[0]
    center_node = ox.distance.nearest_nodes(G, X=p['geometry'].x, Y=p["geometry"].y)

    subgraph = nx.ego_graph(
            G,
            center_node,
            radius= isochrone_time * 60,
            distance="travel_time"
        )

    node_points = [
        Point((data["x"], data["y"]))
        for node, data in subgraph.nodes(data=True)
    ]

    center_point = Point(G.nodes[center_node]["x"], G.nodes[center_node]["y"])

    polygon = gpd.GeoSeries(node_points).union_all().convex_hull
    isochrone_polys.append(
            gpd.GeoDataFrame(
                {"stop_id": [p["stop_id"]], "geometry": [polygon]},
                crs="EPSG:4326"
            )
        )

isochrone_polys = gpd.GeoDataFrame(pd.concat(isochrone_polys, ignore_index=True), crs="EPSG:4326")

In [ ]:
isocores_df_all = all_stops[all_stops['stop_id'].isin(isochrone_polys['stop_id'].unique())].merge(isochrone_polys, on='stop_id', how='left')
isocores_df_all.drop(columns=['geometry_x'], inplace=True)
isocores_df_all.rename(columns={'geometry_y': 'geometry'}, inplace=True)
isocores_df_all.set_geometry('geometry', inplace=True)
isocores_df_all

In [ ]:
isocores_df_all.to_csv('Data/Neigh/isochrones_df-csv',index= False)

In [ ]:
isocores_df_all = pd.read_csv('Data/Neigh/isochrones_df-csv')
isocores_df_all['geometry'] = isocores_df_all['geometry'].apply(wkt.loads)
isocores_df_all = gpd.GeoDataFrame(isocores_df_all,geometry='geometry',crs = 'EPSG:4326')
isocores_df_all

In [ ]:
isos = alt.Chart(isocores_df_all).mark_geoshape(
    stroke="black",
    fill="gray",
    opacity = 0.3
).properties(
    width=600,
    height=600
    
).encode(tooltip=['name'])

destinations = pd.read_csv('Nodes/Destinations.csv')
destinations['geometry'] = destinations['geometry'].apply(wkt.loads)
destinations = gpd.GeoDataFrame(destinations,geometry='geometry',crs = 'EPSG:4326')


point = alt.Chart(destinations[destinations['poi_name'] == 'Zona Universitària']).mark_geoshape(size = 0)

isos + point


In [ ]:
pois = pd.read_csv('Nodes/Destinations.csv')
pois['geometry'] = pois['geometry'].apply(wkt.loads)
pois = gpd.GeoDataFrame(pois,geometry='geometry',crs = 'EPSG:4326')
isocores_df_all_merge = isocores_df_all.copy()
pois.to_crs('EPSG:25831', inplace=True)
isocores_df_all_merge.to_crs('EPSG:25831', inplace=True)
isocores_df_all_merge.rename(columns={'id': 'origin_id','stop_id':'origin_stop_id','name':'origin_stop_name','linia':'linia','stop_type':'origin_stop_type'}, inplace=True)
close_destinations = gpd.sjoin(pois,isocores_df_all_merge, how='inner', predicate='within')
close_destinations.drop(columns = ['index_right','category'],inplace = True)
close_destinations = close_destinations[['origin_id', 'origin_stop_id','origin_stop_name', 'origin_stop_type', 'linia', 'poi_name']]
close_destinations['tram'] = close_destinations['origin_stop_name'] + ' - ' + close_destinations['poi_name']
close_destinations

In [ ]:
# cache_dir = Config().CACHE_DIR
# for f in cache_dir.glob("*.mapdb*"):
#     f.unlink()

In [ ]:
transport_network = r5py.TransportNetwork(
    "Data/Neigh/cataluna-260811.osm.pbf",
    elevation_model =["Data/Neigh/elevacions-terreny-lidar-Catalunya-2m-2008-2011tif1786456535055.tif"]
    )

In [ ]:
itineraries = pd.DataFrame()
process = 0
failed_iteraries = []
all_stops['stop_id'] = all_stops['stop_id'].astype(str)
pois.to_crs("EPSG:4326",inplace=True)

for stop in close_destinations['origin_stop_id'].unique():
    process += 1
    print(f"Processing {process}/{close_destinations['origin_stop_id'].nunique()}")
    stops = close_destinations[close_destinations['origin_stop_id'] == stop]
    stops.drop_duplicates(subset=['origin_stop_id', 'poi_name'], inplace=True)

    for _, row in stops.iterrows():
        origin_id = row['origin_stop_id']
        poi  = row['poi_name']

        # if origin_id  != 342:
        #     continue
        # if destination_id != 2:
        #     continue

        # if process != 2374:
        #     continue
        origins = all_stops[all_stops['stop_id'] == origin_id]
        origins.drop_duplicates(subset=['stop_id'], inplace=True)
        origins.rename(columns={'origin_id': 'id'}, inplace=True)
        destinations = pois[pois['poi_name'] == poi].copy()
        destinations.rename(columns={'poi_name': 'id'}, inplace=True)
        origins = gpd.GeoDataFrame(origins, geometry='geometry', crs="EPSG:4326")
        destinations = gpd.GeoDataFrame(destinations, geometry='geometry', crs="EPSG:4326")
        detailed_itineraries = r5py.DetailedItineraries(
            transport_network,
            origins=origins,
            destinations=destinations,
            transport_modes=[r5py.TransportMode.WALK],
            snap_to_network=True,
            speed_walking = 5,
            departure=datetime.datetime(2026, 5, 13, 9, 30),

        )

        if len(detailed_itineraries) == 0:
            failed_iteraries.append((origin_id, poi))
            continue
        exchange_time = detailed_itineraries['travel_time'].sum()
        itineraries_row = pd.DataFrame({
            'origin_stop_id': [origin_id],
            'poi_name': [poi],
            'time': [exchange_time],
            'geometry': [detailed_itineraries['geometry'].iloc[0]],
        })

        itineraries_row = gpd.GeoDataFrame(itineraries_row, geometry='geometry', crs="EPSG:4326")
        itineraries = pd.concat([itineraries, itineraries_row], ignore_index=True)


itineraries["time"] = itineraries["time"].apply(
    lambda x: f"{int(x.total_seconds() // 60):02d}:{int(x.total_seconds() % 60):02d}"
)


In [ ]:
itineraries_final = close_destinations.merge(itineraries, on=['origin_stop_id', 'poi_name'], how='left')
itineraries_final.drop(columns=['origin_stop_id','origin_stop_type','origin_stop_name'], inplace=True)
itineraries_final.rename(columns={'origin_id':'origen','poi_name':'dest','linia':'last_line'},inplace=True)
itineraries_final = itineraries_final[['origen','dest','tram','last_line','time','geometry']]
itineraries_final.insert(4, 'type', 'Egress')
itineraries_final.insert(6, 'directed', True)
itineraries_final

In [ ]:
itineraries_final.to_csv('Edges/Egress.csv', index=False)

In [ ]:
destinations

In [ ]:
destinations[destinations['id'] == 'Zona Universitària']

In [ ]:
destinations['id'].unique()

In [30]:
zona_u_paths

,origen,dest,tram,last_line,type,time,directed,geometry
0,SB-296,Zona Universitària,Av Doctor Marañón - Baldiri i Reixac - Zona Un...,IU Stop,Egress,02:25,True,"LINESTRING (2.11468 41.38166, 2.11448 41.38177..."
1,SB-1650,Zona Universitària,Av Doctor Marañón - Av de Xile - Zona Universi...,IU Stop,Egress,03:43,True,"LINESTRING (2.11484 41.38123, 2.11443 41.38144..."
2,SB-3233,Zona Universitària,ETS d'Arquitectura - Zona Universitària,IU Stop,Egress,02:56,True,"LINESTRING (2.11401 41.38484, 2.11396 41.38482..."
3,B-V1-296,Zona Universitària,Av Doctor Marañón - Baldiri i Reixac - Zona Un...,V1,Egress,02:25,True,"LINESTRING (2.11468 41.38166, 2.11448 41.38177..."
4,B-113-1650,Zona Universitària,Av Doctor Marañón - Av de Xile - Zona Universi...,113,Egress,03:43,True,"LINESTRING (2.11484 41.38123, 2.11443 41.38144..."
5,B-V1-1650,Zona Universitària,Av Doctor Marañón - Av de Xile - Zona Universi...,V1,Egress,03:43,True,"LINESTRING (2.11484 41.38123, 2.11443 41.38144..."
6,B-H6-3233,Zona Universitària,ETS d'Arquitectura - Zona Universitària,H6,Egress,02:56,True,"LINESTRING (2.11401 41.38484, 2.11396 41.38482..."
7,ST-UNIV,Zona Universitària,Zona Universitària - Zona Universitària,IU Stop,Egress,02:00,True,"LINESTRING (2.11419 41.38466, 2.11443 41.38412..."
8,ST-XILE,Zona Universitària,Avinguda de Xile - Zona Universitària,IU Stop,Egress,05:10,True,"LINESTRING (2.11457 41.38032, 2.11461 41.38037..."
9,T-T1-UNIV,Zona Universitària,Zona Universitària - Zona Universitària,T1,Egress,02:00,True,"LINESTRING (2.11419 41.38466, 2.11443 41.38412..."


In [29]:
itineraries_final = gpd.GeoDataFrame(itineraries_final, geometry='geometry', crs="EPSG:4326")
zona_u_paths = itineraries_final[itineraries_final['dest'] == 'Zona Universitària']
paths_layer = alt.Chart(zona_u_paths).mark_geoshape(
    stroke="red",
    filled = False,
    opacity = 0.5,
    color = 'black'
)
point = alt.Chart(pois[pois['poi_name'] == 'Zona Universitària']).mark_geoshape(size = 0)
paths_layer + point

alt.LayerChart(...)